# PHARVO-beta: POS Add to Cart Test

**Objective:** Verify that an authorized staff user (`rafi`) can search for a medicine (`Brufen`), select the **Strip** unit toggle, and click **Add** to add the medicine to the **Current Sale** cart.

### Test Parameters
- **Medicine Search:** `Brufen`
- **Selected Unit:** `Strip`

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Data ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password
MEDICINE_SEARCH = "Brufen"
UNIT = "Strip"

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print(f"[INFO] Starting Add to Cart Test for '{MEDICINE_SEARCH}' ({UNIT})...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 3: Navigate to POS / Sales module via sidebar
    pos_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'POS / Sales')]")
        )
    )
    pos_nav.click()

    # Verify POS screen title
    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'POS / Sales')]")
        )
    )
    print("[INFO] Navigated to POS / Sales terminal.")

    # Step 4: Search for the target medicine
    pos_search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' and contains(@class, 'pos-input')]")
        )
    )
    pos_search_input.clear()
    pos_search_input.send_keys(MEDICINE_SEARCH)
    print(f"[INFO] Searched for medicine: '{MEDICINE_SEARCH}'")

    # Step 5: Wait for matching row in POS table
    target_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//table[contains(@class, 'pos-table')]//tbody//tr[contains(@class, 'pos-row-tr') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{MEDICINE_SEARCH.lower()}')]]")
        )
    )
    med_name_text = target_row.find_element(By.XPATH, ".//td[1]//span[1]").text
    print(f"[INFO] Located medicine row: '{med_name_text}'")

    # Step 6: Select the requested unit toggle (Strip)
    unit_button = target_row.find_element(
        By.XPATH, f".//button[contains(@class, 'pos-unit-btn') and contains(., '{UNIT}')] | .//button[normalize-space()='{UNIT}' or contains(., '{UNIT}')]"
    )
    unit_button.click()
    print(f"[INFO] Selected unit: '{UNIT}'")

    # Step 7: Click the Add button in the same row
    add_button = target_row.find_element(
        By.XPATH, ".//button[contains(., 'Add')]"
    )
    add_button.click()
    print(f"[INFO] Clicked 'Add' button for '{med_name_text}'.")

    # Step 8: Verify item appears in Current Sale Cart panel
    cart_item_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//tr[contains(@class, 'pos-row') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{MEDICINE_SEARCH.lower()}')]]")
        )
    )

    # Verify cart item name, unit pill, and line item total
    cart_item_name = cart_item_row.find_element(By.XPATH, ".//td[1]/div[1]").text
    unit_badge = cart_item_row.find_element(By.XPATH, ".//td[1]//*[contains(@class, 'pos-pill')]").text
    item_total = cart_item_row.find_element(By.XPATH, ".//td[4]/div[1]").text

    # Verify Cart Header counter badge
    cart_badge = driver.find_element(
        By.XPATH, "//h3[contains(text(), 'Current Sale')]/following-sibling::span[contains(@class, 'pos-pill')]"
    )

    if cart_item_row.is_displayed():
        print("PASS: Item successfully added to POS cart.")
        print(f"      - Cart Item: '{cart_item_name}'")
        print(f"      - Selected Unit & Qty: '{unit_badge}'")
        print(f"      - Line Total: '{item_total}'")
        print(f"      - Active Cart Count Badge: '{cart_badge.text}'")

        assert UNIT.lower() in unit_badge.lower(), f"Expected unit '{UNIT}' in badge, got '{unit_badge}'"
    else:
        print("FAIL: Item row was not visible in the cart table.")

except Exception as error:
    print(f"FAIL: Add to Cart test encountered error: {error}")

finally:
    # Step 9: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
